<div style="border-left: 5px solid #4285f4; padding: 16px 20px; margin: 6px 0 20px 0; border-radius: 6px; background: rgba(66,133,244,0.06);">
<h1>✨ Mejorar imágenes con IA</h1>
<p><strong>Restauración de fotografías con CodeFormer</strong></p>
<p>Un flujo sencillo para recuperar detalles, mejorar rostros y ampliar tus imágenes.</p>
<hr>
<h3>🧭 Cómo usar este cuaderno</h3>
<ol>
<li>⚙️ Preparar el entorno.</li>
<li>📤 Subir una foto o seleccionar una que ya tengas guardada.</li>
<li>🎚️ Ajustar la fidelidad, la ampliación y las mejoras opcionales.</li>
<li>🚀 Restaurar la imagen.</li>
<li>🖼️ Comparar el antes y el después.</li>
<li>📥 Descargar el resultado.</li>
</ol>
<p><strong>💡 Recomendación:</strong> selecciona una GPU en <em>Entorno de ejecución → Cambiar tipo de entorno de ejecución</em> para acelerar el procesamiento. Si Colab reinicia el entorno, vuelve a ejecutar la preparación.</p>
</div>

<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>⚙️ 01 · Preparar el entorno</h2>
</div>

Ejecuta esta celda primero. Prepara las dependencias, conecta Google Drive y comprueba que estén disponibles los cinco modelos necesarios para restaurar imágenes. Los modelos se descargan desde el Release del proyecto y se verifican mediante SHA-256.

<blockquote>⏳ La primera ejecución puede tardar unos minutos mientras se preparan los componentes.</blockquote>

In [ ]:
from pathlib import Path
import os, sys, shutil, subprocess, urllib.request, time
from PIL import Image, ImageOps
from IPython.display import display, Markdown
import matplotlib.pyplot as plt

PROYECTO = Path('/content/Mejorar_Imagenes_con_IA')
ENTRADA = PROYECTO / 'entrada'
RESULTADOS = PROYECTO / 'resultados'
EXTENSIONES = {
    '.png', '.jpg', '.jpeg', '.webp',
    '.bmp', '.tif', '.tiff',
}

# Montar Google Drive si estamos en Colab.
EN_COLAB = False
files = None
try:
    from google.colab import drive, files
    drive.mount('/content/drive', force_remount=False)
    DRIVE_PROYECTO = Path(
        '/content/drive/MyDrive/Mejorar_Imagenes_con_IA'
    )
    DRIVE_ORIGINALES = DRIVE_PROYECTO / 'originales'
    DRIVE_RESULTADOS = DRIVE_PROYECTO / 'resultados'
    DRIVE_ORIGINALES.mkdir(parents=True, exist_ok=True)
    DRIVE_RESULTADOS.mkdir(parents=True, exist_ok=True)
    EN_COLAB = True
except Exception as e:
    print('Aviso de Google Drive/Colab:', repr(e))

for carpeta in (ENTRADA, RESULTADOS):
    carpeta.mkdir(parents=True, exist_ok=True)

# Buscar un repositorio ya preparado antes de descargar uno nuevo.
# Si no existe una copia válida, descargar la rama predeterminada oficial.
candidatos_repo = [
    PROYECTO / 'CodeFormer',
    Path('/content/CodeFormer'),
    Path('/content/drive/MyDrive/Mejorar_Imagenes_con_IA')
    / 'CodeFormer',
]
CODEFORMER = next(
    (p for p in candidatos_repo
     if (p / 'inference_codeformer.py').is_file()),
    None,
)

if CODEFORMER is None:
    CODEFORMER = PROYECTO / 'CodeFormer'
    if CODEFORMER.exists():
        shutil.rmtree(CODEFORMER)
    print('Descargando el repositorio oficial de CodeFormer desde la rama predeterminada…')
    subprocess.run(
        ['git', 'clone', '--depth', '1',
         'https://github.com/sczhou/CodeFormer.git',
         str(CODEFORMER)],
        check=True,
    )
else:
    print('Repositorio encontrado:', CODEFORMER)

# Instalar/comprobar dependencias tanto en un clon nuevo como en uno existente.
# En Colab, los paquetes del entorno no persisten aunque el repositorio esté en Drive.
archivo_requisitos = CODEFORMER / 'requirements.txt'
if not archivo_requisitos.is_file():
    raise FileNotFoundError(
        'No se encontró requirements.txt en el repositorio: '
        + str(archivo_requisitos)
    )
print('Instalando/comprobando dependencias. Puede tardar unos minutos…')
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(archivo_requisitos)],
    check=True,
)

SCRIPT = CODEFORMER / 'inference_codeformer.py'
if not SCRIPT.is_file():
    mensaje = 'No se encontró inference_codeformer.py en '
    raise FileNotFoundError(mensaje + str(CODEFORMER))

# Algunas copias no incluyen version.py.
# BasicSR lo importa al iniciar.
version_py = CODEFORMER / 'basicsr' / 'version.py'
if (CODEFORMER / 'basicsr').is_dir() and not version_py.exists():
    version_py.write_text(
        "__version__ = '0.0.0-local'\n__gitsha__ = 'unknown'\n",
        encoding='utf-8',
    )

# Descargar pesos desde nuestro propio GitHub Release.
# Se verifica SHA-256 para detectar descargas incompletas o archivos alterados.
import hashlib

BASE_URL = (
    'https://github.com/Pablitus666/Mejorar_Imagenes_con_IA/'
    'releases/download/modelos-v1/'
)

PESO_CODEFORMER = (
    CODEFORMER / 'weights' / 'CodeFormer' / 'codeformer.pth'
)
PESO_DETECCION = (
    CODEFORMER / 'weights' / 'facelib'
    / 'detection_Resnet50_Final.pth'
)
PESO_PARSING = CODEFORMER / 'weights' / 'facelib' / 'parsing_parsenet.pth'
PESO_YOLO = CODEFORMER / 'weights' / 'facelib' / 'yolov5l-face.pth'

PESOS = [
    {
        'nombre': 'codeformer.pth',
        'url': BASE_URL + 'codeformer.pth',
        'destino': PESO_CODEFORMER,
        'minimo_bytes': 100_000_000,
        'sha256': '1009e537e0c2a07d4cabce6355f53cb66767cd4b4297ec7a4a64ca4b8a5684b7',
    },
    {
        'nombre': 'detection_Resnet50_Final.pth',
        'url': BASE_URL + 'detection_Resnet50_Final.pth',
        'destino': PESO_DETECCION,
        'minimo_bytes': 1_000_000,
        'sha256': '6d1de9c2944f2ccddca5f5e010ea5ae64a39845a86311af6fdf30841b0a5a16d',
    },
    {
        'nombre': 'parsing_parsenet.pth',
        'url': BASE_URL + 'parsing_parsenet.pth',
        'destino': PESO_PARSING,
        'minimo_bytes': 1_000_000,
        'sha256': '3d558d8d0e42c20224f13cf5a29c79eba2d59913419f945545d8cf7b72920de2',
    },
    {
        'nombre': 'RealESRGAN_x2plus.pth',
        'url': BASE_URL + 'RealESRGAN_x2plus.pth',
        'destino': CODEFORMER / 'weights' / 'realesrgan' / 'RealESRGAN_x2plus.pth',
        'minimo_bytes': 60_000_000,
        'sha256': '49fafd45f8fd7aa8d31ab2a22d14d91b536c34494a5cfe31eb5d89c2fa266abb',
    },
    {
        'nombre': 'yolov5l-face.pth',
        'url': BASE_URL + 'yolov5l-face.pth',
        'destino': PESO_YOLO,
        'minimo_bytes': 100_000_000,
        'sha256': '1ba9d2125fda4d823df5152b9fc2903c59aa76c0d3771e02bcf13a56a282cf96',
    },
]

def calcular_sha256(ruta):
    sha = hashlib.sha256()
    with open(ruta, 'rb') as archivo:
        for bloque in iter(lambda: archivo.read(1024 * 1024), b''):
            sha.update(bloque)
    return sha.hexdigest()

def descargar_peso(url, destino, minimo_bytes, sha256_esperado):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)

    if destino.is_file() and destino.stat().st_size >= minimo_bytes:
        print(f'Verificando peso existente: {destino.name}…')
        sha_existente = calcular_sha256(destino)
        if sha_existente.lower() == sha256_esperado.lower():
            print(f'Peso verificado: {destino.name} (SHA-256 correcto)')
            return
        print(f'El SHA-256 de {destino.name} no coincide; se descargará de nuevo.')

    temporal = destino.with_suffix(destino.suffix + '.part')
    if temporal.exists():
        temporal.unlink()
    print(f'Descargando {destino.name} desde nuestro Release…')
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=90) as respuesta, open(temporal, 'wb') as salida:
            total = int(respuesta.headers.get('Content-Length', '0') or 0)
            descargado = 0
            while True:
                bloque = respuesta.read(1024 * 1024)
                if not bloque:
                    break
                salida.write(bloque)
                descargado += len(bloque)
                if total:
                    print(f'  {descargado / total:.0%}', end='\r')

        tamano = temporal.stat().st_size
        if tamano < minimo_bytes:
            raise RuntimeError(
                f'El archivo descargado es demasiado pequeño ({tamano:,} bytes).'
            )
        sha_descargado = calcular_sha256(temporal)
        if sha_descargado.lower() != sha256_esperado.lower():
            raise RuntimeError(
                'La verificación SHA-256 ha fallado. '
                f'Esperado: {sha256_esperado}; obtenido: {sha_descargado}'
            )
        temporal.replace(destino)
        print(f'Verificado y guardado: {destino.name} ({tamano:,} bytes)')
    except Exception as e:
        if temporal.exists():
            temporal.unlink()
        raise RuntimeError(
            f'No se pudo descargar/verificar {destino.name} desde {url}. Detalle: {e}'
        ) from e

for peso in PESOS:
    descargar_peso(
        peso['url'], peso['destino'], peso['minimo_bytes'], peso['sha256']
    )

# Configurar CodeFormer para usar el peso RealESRGAN local.
# La URL oficial puede variar según la revisión del script (CodeFormer o Real-ESRGAN).
# Sustituimos cualquier URL HTTP(S) que termine en RealESRGAN_x2plus.pth,
# sin depender de la versión del dominio ni de la función que la contiene.
import re

PESO_REALESRGAN = (
    CODEFORMER / 'weights' / 'realesrgan' / 'RealESRGAN_x2plus.pth'
).resolve()
ruta_local_script = str(PESO_REALESRGAN)
contenido_script = SCRIPT.read_text(encoding='utf-8')

# Guardar una copia del script original una sola vez.
copia_seguridad = SCRIPT.with_suffix(SCRIPT.suffix + '.bak')
if not copia_seguridad.exists():
    shutil.copy2(SCRIPT, copia_seguridad)

# Ejemplo de URL oficial que se corrige:
# https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.1/RealESRGAN_x2plus.pth
patron_url_realesrgan = re.compile(
    r"https?://[^\s\"']*RealESRGAN_x2plus\.pth",
    flags=re.IGNORECASE,
)
contenido_corregido, reemplazos = patron_url_realesrgan.subn(
    lambda _m: ruta_local_script,
    contenido_script,
)

if reemplazos:
    SCRIPT.write_text(contenido_corregido, encoding='utf-8')
    print(
        'URL externa de RealESRGAN sustituida por la ruta local '
        f'({reemplazos} reemplazo(s)).'
    )
elif ruta_local_script in contenido_script:
    print('CodeFormer ya estaba configurado para usar el RealESRGAN local.')
else:
    lineas_relevantes = [
        f'{n}: {linea.strip()}'
        for n, linea in enumerate(contenido_script.splitlines(), 1)
        if 'RealESRGAN' in linea or 'model_path' in linea
    ]
    raise RuntimeError(
        'No se encontró la URL de RealESRGAN_x2plus.pth en el script. '
        'No se modificó el script. Líneas relacionadas: '
        + (' | '.join(lineas_relevantes) or '(ninguna)')
    )

# Verificar que la URL externa ya no aparece y que el archivo local sí.
contenido_final = SCRIPT.read_text(encoding='utf-8')
if ruta_local_script not in contenido_final:
    raise RuntimeError(
        'La verificación de la ruta local de RealESRGAN falló: ' + str(SCRIPT)
    )
if patron_url_realesrgan.search(contenido_final):
    raise RuntimeError(
        'Todavía queda una URL externa de RealESRGAN_x2plus.pth en ' + str(SCRIPT)
    )
if not PESO_REALESRGAN.is_file() or PESO_REALESRGAN.stat().st_size == 0:
    raise FileNotFoundError(
        'No existe el peso local de RealESRGAN o está vacío: '
        + str(PESO_REALESRGAN)
    )

# Garantizar que desmarcar "Mejorar también el fondo" desactive
# RealESRGAN incluso si se reutiliza una copia cuyo valor predeterminado
# lo activa. La celda de restauración solo añade --bg_upsampler realesrgan
# cuando la opción está marcada.
patron_argumento_fondo = re.compile(
    r"(?m)^(?P<prefijo>\s*parser\.add_argument\(\s*['\"]--bg_upsampler['\"][^\n]*?\bdefault\s*=\s*)"
    r"['\"](?P<valor>[^'\"]*)['\"]"
)
coincidencia_fondo = patron_argumento_fondo.search(contenido_final)
if coincidencia_fondo and coincidencia_fondo.group('valor').lower() == 'realesrgan':
    contenido_final = patron_argumento_fondo.sub(
        lambda m: m.group('prefijo') + "'None'",
        contenido_final,
        count=1,
    )
    SCRIPT.write_text(contenido_final, encoding='utf-8')
    print('Control de fondo ajustado: RealESRGAN queda desactivado por defecto.')
elif coincidencia_fondo:
    print(
        'Control de fondo verificado: valor predeterminado =',
        repr(coincidencia_fondo.group('valor')),
    )
else:
    # Si no se declara un valor predeterminado explícito, argparse usa None.
    linea_bg = next(
        (linea.strip() for linea in contenido_final.splitlines()
         if '--bg_upsampler' in linea),
        None,
    )
    if linea_bg is None:
        raise RuntimeError(
            'No se encontró el argumento --bg_upsampler en CodeFormer; '
            'no es posible garantizar el control de mejora del fondo.'
        )
    if re.search(r"default\s*=\s*['\"]?realesrgan", linea_bg, re.IGNORECASE):
        raise RuntimeError(
            'CodeFormer activa RealESRGAN por defecto y no se pudo ajustar '
            'automáticamente el argumento --bg_upsampler.'
        )
    print('Control de fondo verificado en el argumento:', linea_bg)

print('Peso RealESRGAN local:', PESO_REALESRGAN)
print('Verificación SHA-256: ya realizada durante la descarga de pesos.')

print('\nPreparación lista.')
print('Proyecto:', PROYECTO)
print('Repositorio CodeFormer:', CODEFORMER)
print('Entrada:', ENTRADA)
print('Resultados:', RESULTADOS)
try:
    import torch
    print('PyTorch:', torch.__version__)
    gpu_nombre = (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available()
        else 'no detectada'
    )
    print(
        'GPU disponible:', torch.cuda.is_available(), '-', gpu_nombre
    )
except Exception as e:
    print('No se pudo comprobar CUDA:', repr(e))

def listar_imagenes():
    return sorted(
        [p for p in ENTRADA.iterdir()
         if p.is_file() and p.suffix.lower() in EXTENSIONES],
        key=lambda p: p.name.lower(),
    )


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>📤 02 · Elegir una fotografía</h2>
</div>

Sube <strong>una imagen por vez</strong> o selecciona una fotografía que ya esté guardada en la carpeta de entrada.

Formatos habituales compatibles: JPG, JPEG, PNG, WEBP, BMP y TIFF.

In [ ]:
import ipywidgets as widgets
from IPython.display import clear_output

boton_subir = widgets.Button(
    description='Subir imagen',
    icon='upload',
    button_style='primary',
    layout=widgets.Layout(width='190px', height='40px'),
)
selector_imagen = widgets.Dropdown(
    options=[], description='Imagen:',
    layout=widgets.Layout(width='95%'),
)
estado_subida = widgets.Output()

def actualizar_selector(seleccion=None):
    imagenes = listar_imagenes()
    selector_imagen.options = [(p.name, str(p)) for p in imagenes]
    if seleccion and any(str(p) == seleccion for p in imagenes):
        selector_imagen.value = seleccion
    elif imagenes and selector_imagen.value not in [
        str(p) for p in imagenes
    ]:
        selector_imagen.value = str(imagenes[-1])
    elif not imagenes:
        selector_imagen.value = None

def subir_una_imagen(_):
    with estado_subida:
        clear_output()
        if not EN_COLAB:
            print(
                'La subida directa está disponible '
                'en Google Colab. '

                'Copia la imagen a:', ENTRADA
            )
            return
        carga = files.upload()
        if not carga:
            print('No se seleccionó ninguna imagen.')
            return
        # Si se eligen varios archivos, solo se guarda el primero.
        nombre, contenido = next(iter(carga.items()))
        ext = Path(nombre).suffix.lower()
        if ext not in EXTENSIONES:
            print(
                'Formato no compatible: '
                f'{ext or "(sin extensión)"}'
            )
            return
        destino = ENTRADA / Path(nombre).name
        destino.write_bytes(contenido)
        if 'DRIVE_ORIGINALES' in globals():
            shutil.copy2(destino, DRIVE_ORIGINALES / destino.name)
        actualizar_selector(str(destino))
        print('Imagen guardada:', destino)

boton_subir.on_click(subir_una_imagen)
actualizar_selector()
display(
    widgets.HBox([boton_subir]), selector_imagen, estado_subida
)


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🎚️ 03 · Ajustar la restauración</h2>
</div>

Personaliza el resultado con los controles de la celda siguiente.

- **Fidelidad:** equilibra la conservación de la fotografía original y la reconstrucción de detalles. Un valor menor permite más cambios; uno mayor prioriza conservar la imagen.
- **Ampliación 2× o 4×:** elige el tamaño final. La opción 4× produce imágenes más grandes y puede tardar más.
- **🌄 Mejora del fondo:** activa Real-ESRGAN para mejorar las zonas que no son rostros.
- **🙂 Mejora adicional de rostros:** activa el procesamiento adicional de los rostros detectados.

💡 **Para empezar:** prueba una fidelidad de 0,7 y ampliación 2×. Activa las mejoras adicionales si buscas más detalle; pueden aumentar el tiempo de procesamiento y el uso de memoria de la GPU.

**Aviso para imágenes exigentes:** la ampliación 4×, especialmente con las dos mejoras activadas, puede superar la memoria disponible. En fotografías de más de 20 megapíxeles, se recomienda empezar por 2×. El cuaderno avisará antes de iniciar, pero no bloqueará el procesamiento.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

control_fidelidad = widgets.FloatSlider(
    value=0.7,
    min=0.0,
    max=1.0,
    step=0.05,
    description='Fidelidad:',
    readout_format='.2f',
    continuous_update=False,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='95%'),
)
control_escala = widgets.Dropdown(
    options=[('2×', 2), ('4×', 4)],
    value=2,
    description='Ampliación:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='300px'),
)
control_fondo = widgets.Checkbox(
    value=False,
    description='Mejorar también el fondo (Real-ESRGAN)',
    indent=False,
    layout=widgets.Layout(width='100%'),
)
control_rostros = widgets.Checkbox(
    value=False,
    description='Aplicar mejora adicional a los rostros',
    indent=False,
    layout=widgets.Layout(width='100%'),
)

panel_mejoras = widgets.VBox(
    [
        widgets.HTML(
            '<b>Opciones que tardan más (opcionales)</b>'
        ),
        control_fondo,
        control_rostros,
    ],
    layout=widgets.Layout(width='100%', gap='6px'),
)
display(control_fidelidad)
display(control_escala)
display(panel_mejoras)


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🚀 04 · Restaurar la imagen</h2>
</div>

Cuando hayas elegido la foto y ajustado los parámetros, ejecuta la celda siguiente.

⏳ Espera a que aparezca el mensaje **«¡Restauración completada!»** antes de continuar.

In [ ]:

from datetime import datetime
import time

ultimo_resultado = None
ultimo_original = None
ultimo_directorio = None

# ------------------------------------------------------------
# 1. Validar la imagen y leer los controles
# ------------------------------------------------------------
if not selector_imagen.value:
    raise ValueError(
        'Primero sube una imagen o selecciona una imagen '
        'en la sección de entrada.'
    )

origen = Path(selector_imagen.value)

if not origen.is_file():
    raise FileNotFoundError(
        f'No existe la imagen seleccionada: {origen}'
    )

fidelidad = float(control_fidelidad.value)
escala = int(control_escala.value)
porcentaje = int(round(fidelidad * 100))

marca = (
    f'restauracion_{origen.stem}_w{porcentaje:02d}_{escala}x'
)

out_dir = RESULTADOS / marca

if out_dir.exists():
    marca += '_' + datetime.now().strftime('%Y%m%d_%H%M%S')
    out_dir = RESULTADOS / marca

out_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Preparar la imagen de entrada para CodeFormer
# La rama predeterminada utiliza los argumentos -i y -o.
# Se conserva una copia PNG temporal para admitir los formatos
# de imagen que Pillow puede abrir.
# ------------------------------------------------------------
carpeta_entrada_codeformer = (
    PROYECTO / '_entrada_codeformer' / origen.name
)

if carpeta_entrada_codeformer.exists():
    shutil.rmtree(carpeta_entrada_codeformer)

carpeta_entrada_codeformer.mkdir(
    parents=True,
    exist_ok=True,
)

# CodeFormer busca archivos JPG/PNG con un patrón glob.
# Convertimos la imagen seleccionada a PNG para admitir
# también entradas JPEG, WEBP, TIFF y otros formatos compatibles
# con Pillow.
nombre_entrada_codeformer = origen.stem + '.png'
imagen_entrada_codeformer = (
    carpeta_entrada_codeformer / nombre_entrada_codeformer
)

with Image.open(origen) as im:
    # Aplicar la orientación EXIF físicamente antes de convertir a PNG.
    # Así CodeFormer recibe la fotografía en la orientación correcta.
    ImageOps.exif_transpose(im).convert('RGB').save(
        imagen_entrada_codeformer,
        format='PNG',
    )

# Se indica explícitamente la carpeta de salida con -o.
carpeta_resultados_codeformer = (
    CODEFORMER / 'results' / f'{origen.stem}_w{porcentaje:02d}_{escala}x'
)

# Borrar únicamente el resultado previo de esta misma ejecución
# lógica para evitar confundir archivos antiguos con el nuevo.
if carpeta_resultados_codeformer.exists():
    shutil.rmtree(carpeta_resultados_codeformer)

# ------------------------------------------------------------
# 3. Construir el comando
# ------------------------------------------------------------
comando = [
    sys.executable,
    str(SCRIPT),
    '-i',
    str(imagen_entrada_codeformer),
    '-o',
    str(carpeta_resultados_codeformer),
    '-w',
    str(fidelidad),
    '-s',
    str(escala),
]

if control_fondo.value:
    comando.extend([
        '--bg_upsampler',
        'realesrgan',
    ])

if control_rostros.value:
    comando.append('--face_upsample')

# ------------------------------------------------------------
# 4. Mostrar la configuración
# ------------------------------------------------------------
print('Imagen:', origen.name)
print('Fidelidad:', fidelidad)
print('Ampliación:', f'{escala}×')
print(
    'Mejora del fondo:',
    'activada' if control_fondo.value else 'desactivada',
)
print(
    'Restauración facial:',
    'activada' if control_rostros.value else 'desactivada',
)

with Image.open(origen) as im:
    ancho_entrada, alto_entrada = im.size

megapixeles = (
    ancho_entrada * alto_entrada
) / 1_000_000

print(
    f'Resolución de entrada: {ancho_entrada} × '
    f'{alto_entrada} ({megapixeles:.1f} MP)'
)

if escala == 4 and megapixeles >= 20:
    print(
        '⚠️ AVISO: la imagen es grande para ampliación 4× '
        'y puede superar la memoria disponible.'
    )
elif escala == 4 and control_fondo.value and control_rostros.value:
    print(
        '⚠️ AVISO: la ampliación 4× con las dos mejoras '
        'puede consumir mucha memoria.'
    )

# ------------------------------------------------------------
# 5. Ejecutar CodeFormer
# ------------------------------------------------------------
print('\nProcesando…')
print('Carpeta de entrada:', carpeta_entrada_codeformer)
print('Salida esperada:', carpeta_resultados_codeformer)

inicio_proceso = time.time()

proceso = subprocess.run(
    comando,
    cwd=str(CODEFORMER),
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

registro = proceso.stdout or ''
print('\n--- Registro de CodeFormer ---')
print(registro if registro.strip() else '(sin salida de texto)')

if proceso.returncode != 0:
    if proceso.returncode == -9:
        raise RuntimeError(
            'CodeFormer fue terminado por el sistema (código -9), '
            'normalmente por falta de memoria. Prueba la ampliación 2×.'
        )

    raise RuntimeError(
        f'CodeFormer terminó con código {proceso.returncode}. '
        'Consulta el registro anterior.'
    )

# ------------------------------------------------------------
# 6. Localizar el resultado esperado
# La rama predeterminada guarda la imagen directamente en la
# carpeta indicada mediante -o.
# ------------------------------------------------------------
# CodeFormer guarda la imagen completa dentro de final_results/.
ruta_final_codeformer = (
    carpeta_resultados_codeformer / 'final_results' / f'{origen.stem}.png'
)

if (
    not ruta_final_codeformer.is_file()
    or ruta_final_codeformer.stat().st_size == 0
):
    # Diagnóstico específico: mostrar qué ha creado realmente
    # el script, en vez de atribuirle una salida inexistente.
    archivos_salida = []

    if carpeta_resultados_codeformer.exists():
        archivos_salida = [
            p for p in carpeta_resultados_codeformer.rglob('*')
            if p.is_file()
        ]

    print('\n--- Diagnóstico de salida ---')
    print('Código de retorno:', proceso.returncode)
    print('Carpeta esperada:', carpeta_resultados_codeformer)
    print('Existe la carpeta:', carpeta_resultados_codeformer.exists())

    if archivos_salida:
        for p in archivos_salida[:50]:
            print(f'{p} ({p.stat().st_size:,} bytes)')
    else:
        print('CodeFormer no ha creado archivos en la carpeta esperada.')

    raise FileNotFoundError(
        'CodeFormer terminó sin producir la imagen final esperada. '
        'El registro y el listado anterior permiten distinguir '
        'un problema de detección facial de un problema de guardado.'
    )

# Verificar que el resultado se puede abrir como imagen.
try:
    with Image.open(ruta_final_codeformer) as im:
        im.verify()
except Exception as e:
    raise RuntimeError(
        f'El archivo de salida existe, pero no es una imagen válida: '
        f'{ruta_final_codeformer}. Detalle: {e}'
    ) from e

# ------------------------------------------------------------
# 7. Copiar a la carpeta de resultados del proyecto
# ------------------------------------------------------------
carpeta_final = out_dir / 'final_results'
carpeta_final.mkdir(parents=True, exist_ok=True)

ultimo_resultado = carpeta_final / ruta_final_codeformer.name
shutil.copy2(ruta_final_codeformer, ultimo_resultado)

ultimo_original = origen
ultimo_directorio = out_dir

# Guardar una copia permanente en Google Drive, si está configurado.
if 'DRIVE_RESULTADOS' in globals():
    carpeta_drive = DRIVE_RESULTADOS / marca
    carpeta_drive.mkdir(parents=True, exist_ok=True)

    copia_drive = carpeta_drive / ultimo_resultado.name
    shutil.copy2(ultimo_resultado, copia_drive)

    print('\nCopia permanente en Drive:', copia_drive)

with Image.open(origen) as im:
    dimensiones_originales = im.size

with Image.open(ultimo_resultado) as im:
    dimensiones_restauradas = im.size

print('\n¡Restauración completada!')
print('Resultado:', ultimo_resultado)
print('Dimensiones originales:', dimensiones_originales)
print('Dimensiones restauradas:', dimensiones_restauradas)


<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>🖼️ 05 · Comparar el antes y el después</h2>
</div>

Revisa las dos imágenes una al lado de la otra para comprobar los cambios y la calidad del resultado.

In [ ]:


from PIL import Image, ImageOps
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import display, HTML
from io import BytesIO
import base64

if (
    'ultimo_resultado' not in globals()
    or ultimo_resultado is None
    or not Path(ultimo_resultado).is_file()
):
    print(
        'Todavía no hay una restauración disponible. '
        'Ejecuta primero la celda de restauración.'
    )
else:
    # Cargar las imágenes respetando su orientación EXIF.
    with Image.open(ultimo_original) as im:
        original_rgb = ImageOps.exif_transpose(im).convert('RGB').copy()

    with Image.open(ultimo_resultado) as im:
        restaurada_rgb = ImageOps.exif_transpose(im).convert('RGB').copy()

    # Crear la comparación.
    fig, axes = plt.subplots(1, 2, figsize=(14, 8))

    axes[0].imshow(original_rgb)
    axes[0].set_title(
        f'Original ({original_rgb.width} × {original_rgb.height})'
    )
    axes[0].axis('off')

    axes[1].imshow(restaurada_rgb)
    axes[1].set_title(
        f'CodeFormer ({restaurada_rgb.width} × {restaurada_rgb.height})'
    )
    axes[1].axis('off')

    fig.tight_layout(pad=2)

    # Convertir la figura a PNG en memoria.
    buffer = BytesIO()
    fig.savefig(
        buffer,
        format='png',
        dpi=120,
        bbox_inches='tight',
        facecolor='white'
    )
    plt.close(fig)

    imagen_base64 = base64.b64encode(
        buffer.getvalue()
    ).decode('utf-8')

    # Centrar la figura completa dentro de la salida de Colab.
    display(HTML(f"""
    <div style="
        width: 100%;
        display: flex;
        justify-content: center;
        align-items: center;
        box-sizing: border-box;
    ">
        <img
            src="data:image/png;base64,{imagen_base64}"
            style="
                display: block;
                max-width: 100%;
                height: auto;
                margin: 0 auto;
            "
        />
    </div>
    """))

    print(
        'Resultado listo para descargar:',
        Path(ultimo_resultado).name
    )



<div style="border-left: 4px solid #4285f4; padding: 10px 16px; margin: 8px 0 16px 0; border-radius: 4px;">
<h2>📥 06 · Descargar la imagen restaurada</h2>
</div>

Ejecuta esta celda cuando estés conforme con el resultado. La imagen restaurada también se conserva en la carpeta de resultados de tu proyecto en Google Drive.

✅ Si todavía no hay una restauración disponible, vuelve a la sección **04 · Restaurar la imagen**.

In [ ]:
if (
    'ultimo_resultado' not in globals()
    or ultimo_resultado is None
    or not Path(ultimo_resultado).is_file()
):
    print(
        'No hay una imagen restaurada para descargar. '
        'Ejecuta primero la restauración y la comparación.'
    )
elif not EN_COLAB:
    print(
        'La imagen restaurada está guardada en:',
        ultimo_resultado,
    )
else:
    print('Descargando:', Path(ultimo_resultado).name)
    files.download(str(ultimo_resultado))
